In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/learningkin/ml-research-paper/paper_4.pdf
/kaggle/input/datasets/learningkin/ml-research-paper/paper_5.pdf
/kaggle/input/datasets/learningkin/ml-research-paper/paper_2.pdf
/kaggle/input/datasets/learningkin/ml-research-paper/paper_1.pdf
/kaggle/input/datasets/learningkin/ml-research-paper/paper_3.pdf


In [2]:
!pip install sentence-transformers==2.7.0
!pip install transformers==4.41.2
!pip install huggingface_hub==0.23.4
!pip install accelerate==0.30.1
!pip install faiss-cpu
!pip install pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.2.3
    Uninstalling sentence-transformers-5.2.3:
      Successfully uninstalled sentence-transformers-5.2.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [3]:
import fitz # PyMUPDF
import os

# folder path 

pdf_folder = "/kaggle/input/datasets/learningkin/ml-research-paper"

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    
    for page in doc:
        text+=page.get_text()
        
    return text 

# text with one files 

files = os.listdir(pdf_folder)

pdf_files = [f for f in files if f.endswith(".pdf")]

print("PDF files: ", pdf_files)

# extract from first PDF


sample_pdf = os.path.join(pdf_folder, pdf_files[0])

text = extract_text_from_pdf(sample_pdf)

print(text[:2000]) # print first 2000 chars 

PDF files:  ['paper_4.pdf', 'paper_5.pdf', 'paper_2.pdf', 'paper_1.pdf', 'paper_3.pdf']
Working in Progress
Unifying Group-Relative and Self-Distillation Policy Opti-
mization via Sample Routing
Gengsheng Li 1,2,∗, Tianyu Yang 1,2,∗, Junfeng Fang 3, Mingyang Song 4, Mao Zheng 4,
Haiyun Guo 1,2, Dan Zhang 3, Jinqiao Wang 1,2,5, Tat-Seng Chua 3
1Foundation Model Research Center, Institute of Automation, Chinese Academy of Sciences
2School of Artificial Intelligence, University of Chinese Academy of Sciences
3National University of Singapore
4Tencent
5Wuhan AI Research
∗Equal contribution
Correspondence: haiyun.guo@nlpr.ia.ac.cn, zhangdan25@nus.edu.sg
Abstract
Reinforcement learning with verifiable rewards (RLVR) has become a stan-
dard paradigm for post-training large language models. While Group
Relative Policy Optimization (GRPO) is widely adopted, its coarse credit
assignment uniformly penalizes failed rollouts, lacking the token-level
focus needed to efficiently address specific devi

In [4]:
# # Extact Text from pdf 

# pdf_path = "/kaggle/input/datasets/learningkin/ml-research-paper/paper_1.pdf"

# def extract_text(pdf_path):
#     doc = fitz.open(pdf_path)
#     text = ""
    
#     for page in doc:
#         text += page.get_text()
        
#     return text

# text = extract_text(pdf_path)

# print(text[:1500])

In [5]:
# Extract Text Page-wise

import fitz # PyMuPDF 

pdf_path = "/kaggle/input/datasets/learningkin/ml-research-paper/paper_1.pdf"

# extract words

def extract_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    
    for i, page in enumerate(doc):
        text = page.get_text()
        pages.append({
            "page_num": i + 1,   
            "text": text
        })
    
    return pages

pages = extract_pages(pdf_path)
print(pages[0]["text"][:500])

Batched Contextual Reinforcement: A Task-Scaling Law for
Efficient Reasoning
Bangji Yang∗1, Hongbo Ma∗2, Jiajun Fan1, Ge Liu1
1University of Illinois Urbana-Champaign
2Tsinghua University
Abstract
Large Language Models (LLMs) employing Chain-of-Thought reasoning achieve strong perfor-
mance but suffer from excessive token consumption that inflates inference costs. Existing efficiency
methods—such as explicit length penalties, difficulty estimators, or multi-stage curricula—either
degrade reasoni


In [6]:
# Clean text

def clean_text(text):
    return " ".join(text.split())


for page in pages:
    page["text"] = clean_text(page["text"])

In [7]:
# Chunks per page 

def chunk_text(text, chunk_size=200 ,overlap=50):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size-overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))
        
    return chunks

In [8]:
# Load the sentence transformer

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    cache_folder="/kaggle/working/")
print("Loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded successfully


In [9]:
## Embedding chunks 

pages = extract_pages(pdf_path)

for page in pages:
    page["text"] = clean_text(page["text"])


all_chunks = []

for page in pages:
    chunks = chunk_text(page["text"])
    all_chunks.extend(chunks)

print("Total chunks:", len(all_chunks))
embeddings = model.encode(all_chunks)
print("Embeddings shape:", len(embeddings))

Total chunks: 150
Embeddings shape: 150


In [10]:
print(embeddings)

[[-0.00855918 -0.03993675  0.04445607 ...  0.02388712 -0.05119217
   0.00987025]
 [-0.02177328 -0.02763492 -0.02275219 ...  0.01492328 -0.07703786
  -0.04441322]
 [ 0.02720784 -0.01876974  0.03865747 ...  0.07694986  0.00658792
   0.0193656 ]
 ...
 [-0.04288042  0.04658873  0.02755925 ... -0.00312185 -0.05328586
  -0.02762482]
 [-0.04892223  0.04020992  0.05586436 ... -0.04070775 -0.09226312
  -0.02210407]
 [ 0.03931019 -0.06998815  0.03070848 ...  0.03182908 -0.05287156
   0.03121968]]


In [11]:
print(len(embeddings))
print(len(embeddings[0]))

150
384


In [12]:
import faiss 
import numpy as np

# convert to numpy array 

embedding_array = np.array(embeddings).astype("float32")

# dimension of vectors

dimension = embedding_array.shape[1]

# Create Faiss index 

index = faiss.IndexFlatL2(dimension)

# add embeddings 

index.add(embedding_array)

print("Total vector in Faiss:", index.ntotal)

Total vector in Faiss: 150


In [13]:
# Save Faiss index

faiss.write_index(index, "faisa_index.bin")

In [14]:
# saving the chunks 
import json 

with open("chunks.json", "w") as f:
    json.dump(all_chunks, f)

In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [16]:
from transformers import pipeline

summarizer = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=256
)

2026-04-11 09:31:44.489782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775899904.884615      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775899904.996906      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775899906.008455      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775899906.008495      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775899906.008499      23 computation_placer.cc:177] computation placer alr

In [17]:
print(generate_answer("Explain machine learning simply"))

Machine learning is a process that learns information from a computer program.


In [18]:
# summarize page 

def summarize_page(text):
    chunks = chunk_text(text)
    
    summaries = []
    
    for chunk in chunks:
        prompt = f"""
        Extract 5-7 important bullet points.
        Do not repeat ideas.

        {chunk}
        """
        
        result = summarizer(
            prompt,
            max_length=256,
            truncation=True
        )[0]["generated_text"]
        
        summaries.extend(result.split("\n"))
    
    return "\n".join(summaries)

In [19]:
# Full pipeline BUT IT PRODUCE GARBAGE WHICH I DON'T WANNA

# pdf_path = "/kaggle/input/datasets/learningkin/ml-research-paper/paper_1.pdf"

# pages = extract_pages(pdf_path)

# first_5_pages = pages[:5]

# for page in first_5_pages:
#     clean = clean_text(page["text"])
    
#     summary = summarize_page(clean)
    
#     print(f"\n===== Page {page['page_num']} =====")
#     print(summary)

In [20]:
# Use Groq so lets install it

!pip install groq

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.0 MB/s eta 0:00:00


In [21]:
from kaggle_secrets import UserSecretsClient
from groq import Groq

# Load API key
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Groq_Api")

client = Groq(api_key=api_key)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant", 
    temperature=0.3,
    max_tokens=512,
    messages=[
        {
            "role": "user",
            "content": "Explain machine learning simply"
        }
    ]
)

print(response.choices[0].message.content)

**What is Machine Learning?**

Machine learning is a way for computers to learn and improve their performance on a task without being explicitly programmed. It's a type of artificial intelligence (AI) that enables computers to make decisions, predictions, or classifications based on data.

**How Does it Work?**

Imagine you're trying to teach a child to recognize different animals. You show them pictures of cats, dogs, and birds, and tell them what each one is. The child looks at the pictures and starts to recognize patterns, such as "cats have whiskers" or "dogs have floppy ears." Over time, the child becomes better at recognizing animals based on these patterns.

Machine learning works in a similar way. A computer is trained on a large dataset of examples, such as images, text, or numbers. The computer looks for patterns and relationships in the data, and uses this information to make predictions or decisions.

**Types of Machine Learning**

There are three main types of machine lear

In [22]:
def summarize_with_groq(chunks):
    text = "\n\n".join(chunks[:6])
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.3,
        max_tokens=1024,
        messages=[
            {
                "role": "system",
                "content": "You are an expert ML researcher who explains research papers clearly to students learning machine learning."
            },
            {
                "role": "user",
                "content": f"""Read this ML research paper and summarize it in exactly this format:

**TL;DR:** (1-2 sentences max, what this paper does in simple words)

**Problem:** (what problem does this paper solve?)

**Method:** (how did they solve it? what technique or model did they use?)

**Results:** (key numbers, accuracy, improvements they achieved)

**Why it matters:** (why should someone learning ML care about this?)

**3 things to remember:**
- 
- 
-

Paper content:
{text}"""
            }
        ]
    )
    
    return response.choices[0].message.content

In [23]:
chunks = chunk_text(clean_text(pages[0]["text"]))

summary = summarize_with_groq(chunks)

print(summary)   

**TL;DR:** This paper introduces a new training method called Batched Contextual Reinforcement (BCR) that enables Large Language Models (LLMs) to reason efficiently without sacrificing accuracy.

**Problem:** The paper solves the problem of excessive token consumption in LLMs that employ Chain-of-Thought reasoning, leading to inflated inference costs.

**Method:** The authors propose a minimalist, single-stage training paradigm that trains the model to solve multiple problems simultaneously within a shared context window, rewarded purely by per-instance accuracy. This approach creates an implicit token budget that allows the model to reason efficiently.

**Results:** The paper achieves several key findings, including:
- A novel task-scaling law that shows per-problem token usage decreases monotonically while accuracy degrades more gracefully than baselines.
- A "free lunch" phenomenon at standard single-problem inference, where BCR reduces token usage by 15.8% to 62.6% while maintainin